# Deeper investigation — Schemas and parsing

**Worked solution** · [All exercises](../../index.html) · [Setup](../../README.md)

## What you’ll learn

- Compare CSV and Parquet reads using an explicit schema.
- Distinguish stored data types from valid business values by testing extra malformed inputs.

Optional. Complete [Exercise 3](../03-validate.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](../00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../../docs/RECOVERY.md).

In [1]:
import sys
from pathlib import Path

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'lab_support/runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

from lab_support import checks as check
from lab_support.arrival_files import publish_arrival
from lab_support.checks import todo
from lab_support.runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path
from lab_support.workspace import Workspace

workspace = Workspace(solutions=True)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 10:50:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="extension-schemas"></a>
## Your task

Read `data/extras/sales.csv` with its header and `raw.schema` into `csv_sales`. Compare its values and schema with Parquet. Would declaring `amount_raw` as a string reject `oops`?

Then read `data/extras/invalid_sales.csv` with that same schema into `extra_raw`. Apply your `clean_sales` and inspect the reasons. These extra rows are separate from the eight core inputs.

In [2]:
csv_sales = (
    spark.read.option("header", True)
    .schema(raw.schema)
    .csv(spark_path(DATA_ROOT / "extras/sales.csv"))
)
extra_raw = (
    spark.read.option("header", True)
    .schema(raw.schema)
    .csv(spark_path(DATA_ROOT / "extras/invalid_sales.csv"))
)
extra_cleaned = clean_sales(extra_raw)
extra_cleaned.select("sale_id", "reject_reason").show(truncate=False)

+-------+----------------------------+
|sale_id|reject_reason               |
+-------+----------------------------+
|x1     |missing product key         |
|x2     |invalid or missing timestamp|
|x3     |NULL                        |
+-------+----------------------------+



In [3]:
check.same_rows(raw, csv_sales)
check.extra_rejects(extra_cleaned)

CSV and Parquet values agree.
Extra parsing checks passed. Discuss whether the business would accept x3.


<details><summary>Hint</summary>

Use `spark.read.option(...).schema(...).csv(...)`. A declared storage type and a business validation rule are different things.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [4]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Session stopped; exercise files are under runs/run-84da968697


Return to [all exercises](../../index.html).